# 05 - Variantes Avanzadas de PCA: Kernel PCA, Incremental PCA y Sparse PCA

## Curso de Machine Learning: Modelos No Supervisados

A pesar de su elegancia matemática, el algoritmo estándar de PCA (`PCA`) enfrenta tres limitaciones insalvables en escenarios avanzados:

1. **Restricción de Linealidad Estricta**: PCA solo busca proyecciones en hiperplanos lineales. Si los datos se distribuyen sobre una variedad no lineal curva (esferas concéntricas, espirales, toroides), PCA aplana y colapsa los grupos sin lograr desenrollar la variedad.
2. **Límite de Memoria RAM (*In-Memory Bottleneck*)**: SVD tradicional requiere cargar la matriz de datos completa en memoria simultáneamente. Para datasets masivos de gigabytes o terabytes (imágenes satelitales, video, flujos en tiempo real), la memoria se satura.
3. **Densidad Completa de Cargas (*Dense Loadings*)**: En PCA clásico, cada componente principal es una combinación lineal de **absolutamente todas las $p$ variables originales**. Si $p = 500$, interpretar un factor que asigna pequeños pesos no nulos a 500 variables es casi imposible para un analista.

En este cuaderno estudiaremos en profundidad las tres variantes algorítmicas de frontera que resuelven estas limitaciones:
- **Kernel PCA (`KernelPCA`)**: Proyección no lineal en espacios de Hilbert de reproducción (*Kernel Trick*).
- **Incremental PCA (`IncrementalPCA`)**: Procesamiento por lotes fuera de memoria (*Out-of-Core Streaming*).
- **Sparse PCA (`SparsePCA`)**: Cargas dispersas mediante regularización $L_1$ (Lasso) para máxima interpretabilidad.

---

### Contenidos
1. No Linealidad y el Truco del Kernel: Fundamentos de **Kernel PCA (KPCA)**
2. Desenredando Variedades No Lineales: RBF Kernel PCA vs. PCA Lineal
3. Procesamiento Fuera de Memoria (*Out-of-Core*): **Incremental PCA (IPCA)** con `partial_fit`
4. Interpretabilidad Extrema: **Sparse PCA** y Regularización $L_1$ sobre las Cargas
5. Comparativa Integral de Variantes de PCA
6. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_circles, make_blobs, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA, IncrementalPCA, SparsePCA

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Variedad No Lineal: PCA Lineal vs. Kernel PCA (KPCA)

Propuesto por Schölkopf, Smola y Müller (1998), **Kernel PCA** aplica el **truco del kernel (*Kernel Trick*)** para mapear implícitamente los datos de entrada a un espacio de características de dimensión infinita $\Phi(x) \in \mathcal{H}$ donde las relaciones no lineales se vuelven lineales:

$$K_{ij} = k(x_i, x_j) = \langle \Phi(x_i), \Phi(x_j) \rangle_{\mathcal{H}}$$

El kernel más utilizado es el **Gaussiano o RBF (Radial Basis Function)**:
$$k_{RBF}(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|_2^2)$$

Demostremos visualmente cómo Kernel PCA desenrolla dos círculos concéntricos entrelazados donde PCA lineal falla por completo.


In [ ]:
# Generación de círculos concéntricos no separables linealmente
X_circ, y_circ = make_circles(n_samples=500, factor=0.3, noise=0.06, random_state=42)

# 1. PCA Lineal Estandar
pca_lin = PCA(n_components=2).fit(X_circ)
X_pca_lin = pca_lin.transform(X_circ)

# 2. Kernel PCA con Kernel RBF (gamma=3.0)
kpca_rbf = KernelPCA(n_components=2, kernel='rbf', gamma=3.0, random_state=42)
X_kpca = kpca_rbf.fit_transform(X_circ)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Espacio Original
axes[0].scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ, cmap='coolwarm', s=35, edgecolors='k')
axes[0].set_title('Espacio Original: Círculos Concéntricos No Lineales', fontsize=11, fontweight='bold')
axes[0].set_xlabel('$x_1$')
axes[0].set_ylabel('$x_2$')
axes[0].set_aspect('equal')

# Panel 2: PCA Lineal (Incapaz de separar los círculos)
axes[1].scatter(X_pca_lin[:, 0], X_pca_lin[:, 1], c=y_circ, cmap='coolwarm', s=35, edgecolors='k')
axes[1].set_title('PCA Lineal: Solo Rota los Ejes\n(Los dos círculos siguen entrelazados)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_aspect('equal')

# Panel 3: Kernel PCA RBF (Desenrollado exitoso)
axes[2].scatter(X_kpca[:, 0], X_kpca[:, 1], c=y_circ, cmap='coolwarm', s=35, edgecolors='k')
axes[2].set_title('Kernel PCA (Kernel RBF, $\gamma=3$):\nSepara Limpiamente las Clases a lo Largo de PC1', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Kernel PC1')
axes[2].set_ylabel('Kernel PC2')

plt.suptitle('Superación de la Restricción de Linealidad con Kernel PCA', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()


## 2. Procesamiento Fuera de Memoria (*Out-of-Core*): `IncrementalPCA`

Cuando el tamaño del conjunto de datos excede la memoria RAM disponible ($n > 10^6$), **`IncrementalPCA`** (Ross et al., 2008) procesa los datos en mini-lotes secuenciales mediante el método `.partial_fit(X_batch)`.

Utiliza una actualización de SVD de bajo rango que conserva únicamente los $k$ autovectores acumulados, consumiendo una memoria estrictamente dependiente de `batch_size` e independiente del número total de filas $n$.


In [ ]:
# Simulación de un dataset masivo procesado por lotes
X_grande, _ = make_blobs(n_samples=50_000, n_features=20, centers=5, random_state=42)
batch_size = 5000

ipca = IncrementalPCA(n_components=5, batch_size=batch_size)

# Entrenamiento por lotes sin cargar todo en memoria a la vez
for i in range(0, X_grande.shape[0], batch_size):
    X_batch = X_grande[i:i + batch_size]
    ipca.partial_fit(X_batch)

# Comparación contra PCA estándar
pca_std = PCA(n_components=5).fit(X_grande)

print("Comparación: PCA Estándar vs Incremental PCA (IPCA):")
print(f"  Varianza explicada acumulada PCA Estándar: {pca_std.explained_variance_ratio_.sum()*100:.2f}%")
print(f"  Varianza explicada acumulada IPCA (Lotes):  {ipca.explained_variance_ratio_.sum()*100:.2f}%")
print(f"  Diferencia en estimación de varianza: {abs(pca_std.explained_variance_ratio_.sum() - ipca.explained_variance_ratio_.sum())*100:.4f}%")


## 3. Interpretabilidad Extrema: `SparsePCA`

En **Sparse PCA** (Zou, Hastie y Tibshirani, 2006), se añade una penalización de norma $L_1$ (Lasso) sobre las cargas de los autovectores:

$$\min_{\mathbf{U}, \mathbf{V}} \frac{1}{2} \| \mathbf{X}_c - \mathbf{U} \mathbf{V}^T \|_F^2 + \alpha \sum_{j=1}^{p} \| \mathbf{v}_j \|_1$$

El resultado es que **la gran mayoría de los coeficientes se fuerzan a ser exactamente cero**, dejando únicamente las 2 o 3 variables verdaderamente críticas que componen cada factor.


In [ ]:
# Comparación de Cargas: PCA Clásico (Denso) vs Sparse PCA (Disperso con ceros exactos)
wine = load_wine(as_frame=True)
X_wine_sc = StandardScaler().fit_transform(wine.data)

pca_denso = PCA(n_components=2, random_state=42).fit(X_wine_sc)
spca = SparsePCA(n_components=2, alpha=1.2, random_state=42).fit(X_wine_sc)

df_comparativa_cargas = pd.DataFrame({
    'Variable': wine.feature_names,
    'Carga_PCA_Denso': pca_denso.components_[0],
    'Carga_Sparse_PCA': spca.components_[0]
})

print("Comparación de Cargas en la Primera Componente:")
display(df_comparativa_cargas.round(3))

n_ceros_spca = np.sum(spca.components_[0] == 0.0)
print(f"Variables anuladas exactamente a 0.00 en Sparse PCA: {n_ceros_spca} de 13 variables.")
print("Esto aísla de forma nítida únicamente las variables químicas protagonistas.")


## 4. Conclusiones

| Variante | Cuándo Utilizarla | Ventaja Principal | Parámetros Clave |
| :--- | :--- | :--- | :--- |
| **`PCA` Estándar** | Reducción general de dimensionalidad tabular ($n, p$ moderados). | Solución global analítica exacta vía SVD ortogonal. | `n_components` |
| **`KernelPCA`** | Variedades no lineales curvas (círculos, lunas, variedades de grafos). | Proyección no lineal en espacios de Hilbert con Kernel Trick. | `kernel`, `gamma`, `degree` |
| **`IncrementalPCA`** | Datasets masivos fuera de memoria RAM ($n > 10^6$, streaming). | Ajuste por mini-lotes con consumo acotado de memoria RAM. | `batch_size` |
| **`SparsePCA`** | Necesidad crítica de interpretabilidad en dominios regulados. | Cargas dispersas con coeficientes exactamente iguales a cero ($L_1$). | `alpha`, `ridge_alpha` |

En el siguiente y último cuaderno, **06_Ejercicios_Practicos.ipynb**, aplicarás todo el flujo integral de PCA sobre un caso de estudio real de alta complejidad.
